# Part 3: Target Variable Distribution
Distribution of ISUP_GG and risk classes.

In [5]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA_PATH = Path("41586_2022_5154_MOESM3_ESM.xlsx")
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)
s1 = pd.read_excel(DATA_PATH, sheet_name="TableS1_clinical_data")
sa = s1[s1["Country"] == "South Africa"].copy()
sa["ISUP_GG"] = pd.to_numeric(sa["ISUP_GG"].replace({"1-2": 2}), errors="coerce")
sa["Risk_class"] = sa["ISUP_GG"].apply(
    lambda x: np.nan if pd.isna(x) else ("High Risk" if x >= 3 else "Low Risk")
)

/Users/tombalay/miniforge3/envs/practical/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/var/folders/jp/y16t99n91ssgb6wzwplx64pc0000gn/T/ipykernel_56751/2319102088.py:12: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  sa["ISUP_GG"] = pd.to_numeric(sa["ISUP_GG"].replace({"1-2": 2}), errors="coerce")


In [6]:
isup_counts = sa["ISUP_GG"].value_counts(dropna=False).sort_index()
risk_counts = sa["Risk_class"].value_counts(dropna=False)
display(isup_counts)
display(risk_counts)

ISUP_GG
0.0     3
1.0     8
2.0    10
3.0    11
4.0    41
5.0    42
NaN     8
Name: count, dtype: int64

Risk_class
High Risk    94
Low Risk     21
NaN           8
Name: count, dtype: int64

In [7]:
table3_data = []
for grade in sorted(sa["ISUP_GG"].dropna().unique()):
    n = (sa["ISUP_GG"] == grade).sum()
    risk = "Low Risk" if grade <= 2 else "High Risk"
    table3_data.append(
        {
            "ISUP_Grade_Group": int(grade),
            "Class": risk,
            "n": n,
            "Percentage": f"{100*n/len(sa):.1f}%",
        }
    )
n_miss = sa["ISUP_GG"].isna().sum()
table3_data.append(
    {
        "ISUP_Grade_Group": "Missing",
        "Class": "N/A",
        "n": n_miss,
        "Percentage": f"{100*n_miss/len(sa):.1f}%",
    }
)
table3_df = pd.DataFrame(table3_data)
display(table3_df)
table3_df.to_csv(FIG_DIR / "table3_isup_distribution.csv", index=False)

,ISUP_Grade_Group,Class,n,Percentage
0,0,Low Risk,3,2.4%
1,1,Low Risk,8,6.5%
2,2,Low Risk,10,8.1%
3,3,High Risk,11,8.9%
4,4,High Risk,41,33.3%
5,5,High Risk,42,34.1%
6,Missing,N/A,8,6.5%


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
grade_data = sa["ISUP_GG"].dropna()
grade_vc = grade_data.value_counts().sort_index()

bar_colors = ["#2980b9" if float(val) <= 2 else "#e74c3c" for val in grade_vc.index]

axes[0].bar(
    grade_vc.index.astype(int).astype(str),
    grade_vc.values,
    color=bar_colors,
)
axes[0].set_title("ISUP Grade Group Distribution (SA Cohort)")
axes[0].set_xlabel("ISUP Grade Group")
axes[0].set_ylabel("Number of Patients")
risk_data = sa["Risk_class"].dropna().value_counts()
axes[1].pie(
    risk_data.values,
    labels=risk_data.index,
    autopct="%1.1f%%",
    startangle=90,
    colors=["#c0392b", "#2980b9"][: len(risk_data)],
)
axes[1].set_title("Binary Risk Classification (SA Cohort)")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig2_isup_distribution.png", dpi=200)
plt.close(fig)